# Profile-RAG — Half 1: Embed

Builds the profile-based FAISS vector DB from training profiles.

**Input:** `data/profile_db/essays/profile_store.jsonl` (built by `profile_build.ipynb`)

**Output:** `data/vector_db/essays_profile/` — FAISS index + metadata, consumed by Half 2.

Pipeline:
1. Load training profiles from `data/profile_db/essays/profile_store.jsonl`
2. Embed training profiles → FAISS index in `data/vector_db/essays_profile/`

In [ ]:
from pathlib import Path
import sys, os, json
from typing import Dict, List

import numpy as np

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.store import ProfileStore
from rag.profiler.prompts import FACETS
from rag.faiss_index import FAISSIndex
from rag.embedder import get_embedding

print("Project root:", project_root)

## Configuration

In [ ]:
# --- Paths ----------------------------------------------------------------
train_profile_db = str(project_root / "data/profile_db/essays")         # built by profile_build.ipynb
vector_db_dir    = str(project_root / "data/vector_db/essays_profile")  # FAISS index — output of this notebook

# --- DB rebuild knob ------------------------------------------------------
force_rebuild_db = False   # set True to wipe vector_db_dir and rebuild from scratch

## Step 1 — Load training profiles

In [ ]:
train_store = ProfileStore(str(Path(train_profile_db) / "profile_store.jsonl"))
train_store.load()
train_entries = [e for e in train_store.get_all() if e.get("valid")]
print(f"Loaded {len(train_entries)} valid training profiles from {train_profile_db}")
if not train_entries:
    raise RuntimeError(
        f"No training profiles found at {train_profile_db}. "
        "Run notebook/data_process/profile_build.ipynb first."
    )

## Step 2 — Build the profile-based vector DB

Embeds the full rendered profile of each training entry using the fine-tuned
SBERT model and stores them in a FAISS index.

Layout written to `vector_db_dir`:
- `vectors.faiss` — float32 embeddings.
- `vectors_meta.jsonl` — per-vector metadata: user_id, trait_labels, parsed profile, embedded text.
- `feature_store.jsonl` — per-user profile in `FeatureStore` format.

In [ ]:
def render_full_profile_text(entry: Dict) -> str:
    """Deterministic rendering of a profile for embedding.
    Uses the ``raw`` field if present, otherwise reconstructs from facets/linguistic.
    """
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


def build_profile_vector_db(entries: List[Dict], output_dir: str, force: bool = False) -> None:
    os.makedirs(output_dir, exist_ok=True)
    index_path = os.path.join(output_dir, "vectors.faiss")
    meta_path  = os.path.join(output_dir, "vectors_meta.jsonl")
    fs_path    = os.path.join(output_dir, "feature_store.jsonl")

    if not force and os.path.exists(index_path) and os.path.exists(meta_path):
        print(f"[profile-db] Already built at {output_dir}; skipping.")
        return

    if force:
        for p in (index_path, meta_path, fs_path):
            if os.path.exists(p):
                os.remove(p)

    print(f"[profile-db] Embedding {len(entries)} profiles -> {output_dir}")
    texts_to_embed = [render_full_profile_text(e) for e in entries]
    embeddings = get_embedding(texts_to_embed)
    embeddings = np.array(embeddings, dtype="float32")

    meta, fs_lines = [], []
    for e, embedded_text in zip(entries, texts_to_embed):
        labels = e.get("trait_labels", {})
        rag_labels = {}
        for code, name in [("cOPN", "Openness to Experience"), ("cCON", "Conscientiousness"),
                           ("cEXT", "Extraversion"), ("cAGR", "Agreeableness"), ("cNEU", "Neuroticism")]:
            if code in labels and labels[code] in ("high", "low"):
                rag_labels[name] = labels[code]
        features = {"profile": {"facets": e.get("facets", {}), "linguistic": e.get("linguistic", {})}}
        meta.append({
            "user_id":      e["user_id"],
            "posts_raw":    embedded_text,
            "trait_labels": rag_labels,
            "features":     features,
        })
        fs_lines.append({
            "user_id":      e["user_id"],
            "trait_labels": rag_labels,
            "features":     features,
        })

    idx = FAISSIndex(dimension=embeddings.shape[1])
    idx.build(embeddings, meta)
    idx.save(index_path, meta_path)
    with open(fs_path, "w", encoding="utf-8") as f:
        for line in fs_lines:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")
    print(f"[profile-db] Saved index ({embeddings.shape[0]} vectors, dim={embeddings.shape[1]}).")


build_profile_vector_db(train_entries, vector_db_dir, force=force_rebuild_db)

## Done — outputs for Half 2

Download `data/vector_db/essays_profile/` and upload it to Kaggle as a dataset.
Half 2 will load it directly without re-embedding.

| File | Contents |
|---|---|
| `vectors.faiss` | FAISS index (float32 embeddings) |
| `vectors_meta.jsonl` | Per-vector metadata: user_id, trait_labels, profile |
| `feature_store.jsonl` | Per-user profile in FeatureStore format |

In [ ]:
# Quick sanity: confirm all three files exist
for fname in ("vectors.faiss", "vectors_meta.jsonl", "feature_store.jsonl"):
    p = Path(vector_db_dir) / fname
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {p}")